In [21]:
!pip install -q langchain langchain-community langchain-huggingface faiss-cpu pymupdf sentence-transformers

In [22]:
import os
from typing import List, Dict, Tuple
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

class ResearchGPTEngine:
    def __init__(self, embedding_model: str = "BAAI/bge-small-en-v1.5"):
        print(f"Loading {embedding_model} embeddings...")
        self.embeddings = HuggingFaceEmbeddings(
            model_name=embedding_model,
            encode_kwargs={'normalize_embeddings': True}
        )
        self.vector_store = None
        self.text_splitter = RecursiveCharacterTextSplitter(
            chunk_size=1000,
            chunk_overlap=200
        )

    def build_vector_index(self, documents: List[Document]):
        print("Indexing documents into FAISS vector database...")
        self.vector_store = FAISS.from_documents(documents, self.embeddings)
        print("Vector store built and ready.")

    def retrieve_context(self, query: str, k: int = 3) -> List[Document]:
        if not self.vector_store:
            raise ValueError("The vector store is empty. Please run build_vector_index first.")
        return self.vector_store.similarity_search(query, k=k)

    def format_context_with_citations(self, docs: List[Document]) -> Tuple[str, List[Dict]]:
        blocks = []
        citations = []

        for i, doc in enumerate(docs):
            source_file = doc.metadata.get("source", "Unknown Document")
            page_num = doc.metadata.get("page", "Unknown Page")

            blocks.append(f"[Source {i+1}]: {source_file} (Page {page_num})\nContent: {doc.page_content}")

            citations.append({
                "id": i + 1,
                "source": source_file,
                "page": page_num,
                "snippet": doc.page_content[:120] + "..."
            })

        return "\n\n".join(blocks), citations

print("Engine successfully compiled!")

Engine successfully compiled!


In [23]:
engine = ResearchGPTEngine(embedding_model="BAAI/bge-small-en-v1.5")

documents = [
    Document(
        page_content="Large Language Models (LLMs) often struggle with context window exhaustion. Our architecture uses a dynamic vector routing layer that lowers computational complexity from O(N^2) to O(N log N).",
        metadata={"source": "neural_ctx_2026.pdf", "page": 1}
    ),
    Document(
        page_content="We implement an asymmetric embedding scheme using a specialized BGE transformer topology. Experiments show an accuracy retention rate of 94.2% under dense cross-attention pressure.",
        metadata={"source": "neural_ctx_2026.pdf", "page": 2}
    ),
    Document(
        page_content="By leveraging localized FAISS instances within micro-clusters, enterprise knowledge bases can achieve query response latencies under 45ms.",
        metadata={"source": "neural_ctx_2026.pdf", "page": 3}
    )
]

engine.build_vector_index(documents)

Loading BAAI/bge-small-en-v1.5 embeddings...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Indexing documents into FAISS vector database...
Vector store built and ready.


In [24]:
import json

RAG_SYSTEM_PROMPT = """You are a helpful research assistant.
Answer the question using ONLY the provided text blocks below.
For every claim you make, you must cite the source and page number from the context blocks (e.g., [neural_ctx_2026.pdf, Page 1]).
If the context doesn't contain the answer, say "I cannot find this information in the provided text."

Context:
{context}

Question: {question}
Answer with Citations:"""

def ask_research_assistant(user_query: str, engine_instance: ResearchGPTEngine) -> dict:
    matched_chunks = engine_instance.retrieve_context(user_query, k=2)
    formatted_context, citations_list = engine_instance.format_context_with_citations(matched_chunks)
    final_prompt = RAG_SYSTEM_PROMPT.format(context=formatted_context, question=user_query)

    return {
        "prompt_ready_for_llm": final_prompt,
        "frontend_metadata_citations": citations_list
    }

user_question = "What is the computational complexity of the routing layer?"
output_payload = ask_research_assistant(user_question, engine)

print("\n=== PIPELINE RUN SUCCESSFUL ===")
print("\n--- Compiled Prompt Ready for the LLM ---")
print(output_payload["prompt_ready_for_llm"])

print("\n--- JSON Metadata Output (Ready for Frontend UI) ---")
print(json.dumps(output_payload["frontend_metadata_citations"], indent=2))


=== PIPELINE RUN SUCCESSFUL ===

--- Compiled Prompt Ready for the LLM ---
You are a helpful research assistant. 
Answer the question using ONLY the provided text blocks below. 
For every claim you make, you must cite the source and page number from the context blocks (e.g., [neural_ctx_2026.pdf, Page 1]).
If the context doesn't contain the answer, say "I cannot find this information in the provided text."

Context:
[Source 1]: neural_ctx_2026.pdf (Page 1)
Content: Large Language Models (LLMs) often struggle with context window exhaustion. Our architecture uses a dynamic vector routing layer that lowers computational complexity from O(N^2) to O(N log N).

[Source 2]: neural_ctx_2026.pdf (Page 3)
Content: By leveraging localized FAISS instances within micro-clusters, enterprise knowledge bases can achieve query response latencies under 45ms.

Question: What is the computational complexity of the routing layer?
Answer with Citations:

--- JSON Metadata Output (Ready for Frontend UI) ---